# Ingest Results JSON Files

This notebook demonstrates the ingestion of results JSON files (one per season) into the bronze layer.

## What this notebook covers
* Load the results source files (one JSON per season)
* Define and enforce schema
* Add ingestion metadata columns: source_file and ingestion_timestamp
* Write the transformed data to the bronze Delta table
* Validate the loaded results

This uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source folder and target table
source_folder = f"{landing_folder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
# Preview source folder path
source_folder

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

results_schema = StructType([
    StructField("constructorId", StringType()),
    StructField("date",          StringType()),
    StructField("driverId",      StringType()),
    StructField("grid",          IntegerType()),
    StructField("laps",          IntegerType()),
    StructField("number",        IntegerType()),
    StructField("points",        DoubleType()),
    StructField("position",      IntegerType()),
    StructField("positionText",  StringType()),
    StructField("raceName",      StringType()),
    StructField("round",         IntegerType()),
    StructField("season",        IntegerType()),
    StructField("status",        StringType()),
    StructField("url",           StringType())
])

In [0]:
# Read all results JSON files from the directory into a DataFrame
results_df = (
    spark.read
    .format("json")
    .schema(results_schema)
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .load(source_folder)
)

display(results_df)

In [0]:
# Add ingestion metadata columns
results_final_df = add_ingestion_metadata(results_df)
display(results_final_df)

In [0]:
# Write results data to the bronze Delta table
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze results table
SELECT * FROM formula1.bronze.results

In [0]:
# Display the bronze results table
display(spark.table(table_name))